In [ ]:
# 5A: bağlam uzunluğu etkisi kısmı için her çeşit sonuç üretiliyor. ragas sonuçları elde edildikten sonra çalıştırılmalıdır.

import os
import pandas as pd
import json
import matplotlib.pyplot as plt
import numpy as np

def preprocess_ragas_json(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        raw_data = json.load(f)

    processed_data = []
    for entry in raw_data:
        flat_entry = {
            "index": entry["index"],
            "question": entry["question"][0],
            "ground_truth": entry["ground_truth"][0],
            "answer": entry["answer"][0],
        }
        for metric, value in entry["evaluation_result"].items():
            flat_entry[metric] = value[0] if isinstance(value, list) else value

        flat_entry["context_length"] = len(entry.get("retrieved_contexts", [[]])[0])
        processed_data.append(flat_entry)

    return pd.DataFrame(processed_data)


def calculate_statistics_all(data, metrics):
    stats = {}
    for metric in metrics:
        stats[metric] = {
            "general_average": data[metric].mean(),
            "non_zero_average": data.loc[data[metric] > 0, metric].mean()
        }
    return stats


def calculate_statistics_single_metric(data, metric):
    return {
        "general_average": data[metric].mean(),
        "non_zero_average": data.loc[data[metric] > 0, metric].mean()
    }


def analyze_model(data, metrics, context_lengths):
    results = {}

    results["overall"] = calculate_statistics_all(data, metrics)

    for length in context_lengths:
        length_data = data[data["context_length"] == length]
        results[f"context_length_{length}"] = calculate_statistics_all(length_data, metrics)

    best_n_values = [50, 100, 200]
    for top_n in best_n_values:
        for metric in metrics:
            top_data = data.sort_values(by=metric, ascending=False).head(top_n)
            single_metric_stats = calculate_statistics_single_metric(top_data, metric)
            results[f"best_{top_n}_{metric}"] = single_metric_stats

    for length in context_lengths:
        length_subset = data[data["context_length"] == length]
        for metric in metrics:
            top_50_len = length_subset.sort_values(by=metric, ascending=False).head(50)
            single_metric_stats = calculate_statistics_single_metric(top_50_len, metric)
            results[f"best_50_context_length_{length}_{metric}"] = single_metric_stats

    return results


def compare_models(cosmos_data, gemma_data, metrics):
    comparison = {}
    min_len = min(len(cosmos_data), len(gemma_data))
    cosmos_data_aligned = cosmos_data.head(min_len)
    gemma_data_aligned = gemma_data.head(min_len)

    for metric in metrics:
        comparison[metric] = {
            "cosmos_wins": (cosmos_data_aligned[metric] > gemma_data_aligned[metric]).sum(),
            "gemma_wins": (gemma_data_aligned[metric] > cosmos_data_aligned[metric]).sum(),
            "ties": (cosmos_data_aligned[metric] == gemma_data_aligned[metric]).sum(),
        }
    return comparison


def plot_distributions(cosmos_data, gemma_data, metrics, output_folder="output"):
    for metric in metrics:
        plt.figure()
        plt.hist(cosmos_data[metric], bins=30, alpha=0.5, label="Cosmos", density=False)
        plt.hist(gemma_data[metric], bins=30, alpha=0.5, label="Gemma", density=False)
        plt.title(f"Distribution of {metric}")
        plt.xlabel(metric)
        plt.ylabel("Count")
        plt.legend()
        plot_path = os.path.join(output_folder, f"{metric}_distribution.png")
        plt.savefig(plot_path)
        plt.close()


def convert_numpy(obj):
    if isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {k: convert_numpy(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_numpy(item) for item in obj]
    else:
        return obj


if __name__ == "__main__":
    os.makedirs("output", exist_ok=True)

    cosmos_data = preprocess_ragas_json("cosmos_ragas.json")
    gemma_data = preprocess_ragas_json("gemma_ragas.json")

    cosmos_data.to_csv("output/processed_cosmos_ragas.csv", index=False)
    gemma_data.to_csv("output/processed_gemma_ragas.csv", index=False)

    metrics = ["answer_correctness", "faithfulness", "answer_relevancy"]
    context_lengths = [1, 5, 10, 15]

    cosmos_results = analyze_model(cosmos_data, metrics, context_lengths)
    gemma_results = analyze_model(gemma_data, metrics, context_lengths)

    comparison_results = compare_models(cosmos_data, gemma_data, metrics)

    plot_distributions(cosmos_data, gemma_data, metrics, output_folder="output")

    with open("output/cosmos_analysis.json", "w") as f:
        json.dump(cosmos_results, f, indent=4)

    with open("output/gemma_analysis.json", "w") as f:
        json.dump(gemma_results, f, indent=4)

    comparison_results_serializable = convert_numpy(comparison_results)
    with open("output/comparison_results.json", "w") as f:
        json.dump(comparison_results_serializable, f, indent=4)

    print("Analysis complete. Results saved to 'output' folder.")


Analysis complete. Results saved to 'output' folder.


In [9]:
# verilen index için sonuçları gösteren kod

import json

file_path = "cosmos_ragas.json"
index_to_find = 5

with open(file_path, "r", encoding="utf-8") as f:
    data = json.load(f)

entry = next((item for item in data if item["index"] == index_to_find), None)

if entry is None:
    print(f"No entry found with index {index_to_find}.")
else:
    print(f"Index: {entry['index']}\n")
    print("Evaluation Results:")
    for metric, values in entry["evaluation_result"].items():
        print(f"  {metric}: {values[0]}")

    print("\nQuestion:")
    print(entry["question"][0])

    print("\nGround Truth:")
    print(entry["ground_truth"][0])

    print("\nAnswer:")
    print(entry["answer"][0])

    print("\nRetrieved Contexts:")
    for context in entry.get("retrieved_contexts", [[]])[0]:
        print(f"  - {context}")


Index: 5

Evaluation Results:
  answer_correctness: 0.44001675767731147
  faithfulness: 0.8
  answer_relevancy: 0.8447064416719368

Question:
.bm uzantısının kullanımı ne zaman başladı?

Ground Truth:
.bm uzantısı 2007 yılında kullanıma açıldı.

Answer:
uzantısının kullanımı ne zaman başladı?

Cevap:.bm uzantısının kullanımı 2007 yılında kullanıma açıldı. Bu uzantı, Bermuda'ya ait internet ülke alan adıdır ve 1993 yılında Bermuda Koleji tarafından istendi. 2007 yılında kullanıma açılmasından bu yana.bm uzantısını kullanarak internet alan adı oluşturmak mümkün hale gelmiştir.

Retrieved Contexts:
  - Franklin Delano Roosevelt (/ˈroʊzəvəlt/, /-vɛlt/ ROH-zə-velt; 30 Ocak 1882 - 12 Nisan 1945), adının ve soyadının baş harfleri FDR ile de anılır, 1933 yılından 1945'teki ölümüne kadar Amerika Birleşik Devletleri'nin otuz ikinci başkanı olarak görev yapan Amerikan politikacı ve devlet adamı. Demokrat Parti üyesi olarak dört başkanlık seçimi kazandı, Büyük Buhran ve II. Dünya Savaşı'nın da yaş

In [3]:
# 5B: doğru metin konumu kısmı için her çeşit sonuç üretiliyor. ragas sonuçları elde edildikten sonra çalıştırılmalıdır.

import os
import pandas as pd
import json
import matplotlib.pyplot as plt
import numpy as np

def preprocess_ragas_json(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        raw_data = json.load(f)

    processed_data = []
    for entry in raw_data:
        flat_entry = {
            "index": entry["index"],
            "question": entry["question"][0],
            "ground_truth": entry["ground_truth"][0],
            "answer": entry["answer"][0],
        }
        for metric, value in entry["evaluation_result"].items():
            flat_entry[metric] = value[0] if isinstance(value, list) else value

        flat_entry["position"] = entry["index"] % 15
        processed_data.append(flat_entry)

    return pd.DataFrame(processed_data)


def calculate_statistics_all(data, metrics):
    stats = {}
    for metric in metrics:
        stats[metric] = {
            "general_average": data[metric].mean(),
            "non_zero_average": data.loc[data[metric] > 0, metric].mean()
        }
    return stats

def calculate_statistics_single_metric(data, metric):
    return {
        "general_average": data[metric].mean(),
        "non_zero_average": data.loc[data[metric] > 0, metric].mean()
    }

def analyze_model(data, metrics, positions):
    results = {}

    results["overall"] = calculate_statistics_all(data, metrics)

    for position in positions:
        position_data = data[data["position"] == position]
        results[f"position_{position}"] = calculate_statistics_all(position_data, metrics)

    best_n_values = [50, 100, 200]
    for top_n in best_n_values:
        for metric in metrics:
            top_data = data.sort_values(by=metric, ascending=False).head(top_n)
            single_metric_stats = calculate_statistics_single_metric(top_data, metric)
            results[f"best_{top_n}_{metric}"] = single_metric_stats

    return results

def compare_models(cosmos_data, gemma_data, metrics):
    comparison = {}
    min_len = min(len(cosmos_data), len(gemma_data))
    cosmos_data_aligned = cosmos_data.head(min_len)
    gemma_data_aligned = gemma_data.head(min_len)

    for metric in metrics:
        comparison[metric] = {
            "cosmos_wins": (cosmos_data_aligned[metric] > gemma_data_aligned[metric]).sum(),
            "gemma_wins": (gemma_data_aligned[metric] > cosmos_data_aligned[metric]).sum(),
            "ties": (cosmos_data_aligned[metric] == gemma_data_aligned[metric]).sum(),
        }
    return comparison

def plot_distributions(cosmos_data, gemma_data, metrics, output_folder="output"):
    for metric in metrics:
        plt.figure()
        plt.hist(cosmos_data[metric], bins=30, alpha=0.5, label="Cosmos", density=False)
        plt.hist(gemma_data[metric], bins=30, alpha=0.5, label="Gemma", density=False)
        plt.title(f"Distribution of {metric}")
        plt.xlabel(metric)
        plt.ylabel("Count")
        plt.legend()
        plot_path = os.path.join(output_folder, f"{metric}_distribution.png")
        plt.savefig(plot_path)
        plt.close()

def convert_numpy(obj):
    if isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {k: convert_numpy(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_numpy(item) for item in obj]
    else:
        return obj

if __name__ == "__main__":
    os.makedirs("outputb", exist_ok=True)

    cosmos_data = preprocess_ragas_json("cosmos_b_ragas_final.json")
    gemma_data = preprocess_ragas_json("gemma_b_ragas_final.json")

    cosmos_data.to_csv("outputb/cosmos_b_ragas_final.csv", index=False)
    gemma_data.to_csv("outputb/gemma_b_ragas_final.csv", index=False)

    metrics = ["answer_correctness", "faithfulness", "answer_relevancy"]
    positions = list(range(15))

    cosmos_results = analyze_model(cosmos_data, metrics, positions)
    gemma_results = analyze_model(gemma_data, metrics, positions)

    comparison_results = compare_models(cosmos_data, gemma_data, metrics)

    plot_distributions(cosmos_data, gemma_data, metrics, output_folder="outputb")

    with open("outputb/cosmos_analysis.json", "w") as f:
        json.dump(cosmos_results, f, indent=4)

    with open("outputb/gemma_analysis.json", "w") as f:
        json.dump(gemma_results, f, indent=4)

    comparison_results_serializable = convert_numpy(comparison_results)
    with open("outputb/comparison_results.json", "w") as f:
        json.dump(comparison_results_serializable, f, indent=4)

    print("Analysis complete. Results saved to 'outputb' folder.")


Analysis complete. Results saved to 'outputb' folder.
